In [87]:
import pandas as pd
import os

In [88]:
telco = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print(telco.shape)

telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
telco.dropna(subset='TotalCharges', inplace=True)
telco.reset_index(drop=True, inplace=True)
print(telco.shape)


(7043, 21)
(7032, 21)


In [89]:
print(f"Columns count before dropping customerID: {telco.shape[1]}")

telco.drop(columns='customerID', inplace=True)
print(f"Columns count after dropping customerID: {telco.shape[1]}")

Columns count before dropping customerID: 21
Columns count after dropping customerID: 20


In [90]:
cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

for col in telco[cols]:
    telco[col] = telco[col].map({'Yes': 1, 'No': 0})
    print(telco[col].value_counts())
    print('\n -------------------')



Partner
0    3639
1    3393
Name: count, dtype: int64

 -------------------
Dependents
0    4933
1    2099
Name: count, dtype: int64

 -------------------
PhoneService
1    6352
0     680
Name: count, dtype: int64

 -------------------
PaperlessBilling
1    4168
0    2864
Name: count, dtype: int64

 -------------------
Churn
0    5163
1    1869
Name: count, dtype: int64

 -------------------


In [91]:
cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in telco[cols]:
    telco[col] = telco[col].map({
        'Yes': 1,
        'No': 0,
        'No internet service': 0,
        'No phone service': 0
    })
    print(telco[col].value_counts())
    print('\n -------------------')

MultipleLines
0    4065
1    2967
Name: count, dtype: int64

 -------------------
OnlineSecurity
0    5017
1    2015
Name: count, dtype: int64

 -------------------
OnlineBackup
0    4607
1    2425
Name: count, dtype: int64

 -------------------
DeviceProtection
0    4614
1    2418
Name: count, dtype: int64

 -------------------
TechSupport
0    4992
1    2040
Name: count, dtype: int64

 -------------------
StreamingTV
0    4329
1    2703
Name: count, dtype: int64

 -------------------
StreamingMovies
0    4301
1    2731
Name: count, dtype: int64

 -------------------


In [92]:

telco['gender'] = telco['gender'].map({'Male': 1, 'Female': 0})
print(telco['gender'].value_counts())


gender
1    3549
0    3483
Name: count, dtype: int64


In [93]:
telco = pd.get_dummies(
    telco,
    columns=['InternetService', 'Contract', 'PaymentMethod'],
    drop_first=False, 
    dtype=int
)

print(telco.shape)
print(telco.columns.tolist())

(7032, 27)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [94]:
# Make Churn the last column on the df
cols = [c for c in telco.columns if c != 'Churn'] + ['Churn']
telco = telco[cols]

In [95]:
print('--- Final telco cleanned overview ----')
print(f'Rows:    {telco.shape[0]:,}')
print(f'Columns: {telco.shape[1]}')
print(f'Missing values: {telco.isnull().sum().sum()}')
print(f'\nColumns dtypes:\n{telco.dtypes}')

--- Final telco cleanned overview ----
Rows:    7,032
Columns: 27
Missing values: 0

Columns dtypes:
gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
InternetService_DSL                          int64
InternetService_Fiber optic     

In [96]:
os.makedirs('../data/processed', exist_ok=True)
telco.to_csv('../data/processed/telco_clean.csv', index=False)

print('Saved to ../data/processed/telco_clean.csv')
print(f'File size: {os.path.getsize('../data/processed/telco_clean.csv')}')

Saved to ../data/processed/telco_clean.csv
File size: 452945


## Data Cleaning Summary

**Input:** `data/raw/Telco-Customer-Churn.csv` — 7,043 rows, 21 columns  
**Output:** `data/processed/telco_clean.csv` — 7,032 rows, 27 columns

| Step | Column(s) | Issue | Decision |
|------|-----------|-------|----------|
| Drop rows | TotalCharges | 11 blank strings hidden as empty — not caught by `.isnull()` | Converted to numeric with `errors='coerce'`, dropped 11 NaN rows. All had tenure = 0 — imputing billing for unbilled customers is fabrication |
| Drop column | customerID | Identifier — no predictive signal | Dropped |
| Binary encoding | Partner, Dependents, PhoneService, PaperlessBilling, Churn | Stored as Yes/No strings | Mapped Yes → 1, No → 0 |
| Collapse to binary | MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | 3-value columns: Yes / No / No internet service (or No phone service) | Mapped Yes → 1, everything else → 0 |
| Binary encoding | gender | Male/Female string | Mapped Male → 1, Female → 0 |
| One-hot encoding | InternetService, Contract, PaymentMethod | Multi-class categoricals with no ordinal relationship | `pd.get_dummies(drop_first=False, dtype=int)` — expands to 10 new columns |
| Column reorder | Churn | Target column was at index 16 | Moved to last position for clean `X / y` splits in modelling |

**Final dataset:** 7,032 rows · 27 columns · 0 missing values · all int64/float64 · Churn is last column